# PEMS — Notebook 1: Stack Navigation and Indirection

**Goal:** Understand how one navigation loop can run every menu in PEMS. Two ideas make that possible:

- **A stack** remembers the route through the menus. `append()` moves forward, `pop()` goes back, and `stack[-1]` names the current menu.
- **Menus as data.** A dictionary holds each menu's title, choices, and destinations, so the loop never needs menu-specific code.

You will **not** write the navigation loop yourself. You **will** trace it, explain it, and extend PEMS by changing only the menu data.

This notebook covers **navigation only**. Choices that don't open a submenu do nothing yet; Notebook 2 adds actions.

**How to work:** Run the cells in order. Predict before you run. Interactive calls are commented out so **Kernel → Restart & Run All** never stops for input.

---
## 1. The problem: navigation without menu data

Here is a working navigation loop that uses a stack but hard-codes every menu. Read it; you don't need to run it.

In [ ]:
# NAVIGATION WITHOUT MENU DATA — every menu is a special case.

def run_hardcoded_navigation():
    stack = ["client_management"]

    while stack:

        # FIRST set of special cases: what does each menu display?
        if stack[-1] == "client_management":
            print("\nClient Management Menu")
            print("1) Select Client")
            print("2) Create Client")
        elif stack[-1] == "client_workspace":
            print("\nClient Workspace")
            print("1) Transactions")
            print("2) Reports")
        elif stack[-1] == "transactions":
            print("\nTransactions")
            print("1) Contribution")
            print("2) Withdrawal")
        elif stack[-1] == "reports":
            print("\nReports")
            print("1) Holdings")
            print("2) Performance")

        choice = input("Choice (Enter to go back): ").strip()

        if choice == "":
            stack.pop()
            continue

        # SECOND set of special cases: what does each choice do?
        if stack[-1] == "client_management":
            if choice == "1":
                stack.append("client_workspace")
            elif choice == "2":
                print("Create Client — no action yet")
            else:
                print("Invalid choice")
        elif stack[-1] == "client_workspace":
            if choice == "1":
                stack.append("transactions")
            elif choice == "2":
                stack.append("reports")
            else:
                print("Invalid choice")
        elif stack[-1] == "transactions":
            if choice in ("1", "2"):
                print("\nNo action yet\n")
            else:
                print("Invalid choice")
        elif stack[-1] == "reports":
            if choice in ("1", "2"):
                print("\nNo action yet\n")
            else:
                print("Invalid choice")

    print("Navigation finished.")

# OPTIONAL: uncomment to try it.
# run_hardcoded_navigation()

The *process* is the same for every menu: display it, read a choice, then move forward, stay, or go back. Only the **titles, choices, and destinations** differ — yet each lives in Python code, often in two places. With twenty menus, the special cases multiply.

**Central question:** What stays the same across menus, and what changes?

In [ ]:
# QUESTION 1
# If we add a third option to Reports, which TWO parts of
# run_hardcoded_navigation() must change? Why two?

# YOUR ANSWER:

---
## 2. Move the differences into data

Each menu becomes a dictionary with a `title` and a list of `options`. Each option has a label (`menu_item`) and a destination (`next_menu`):

- A string such as `"transactions"` is the **name** of the menu to open.
- `None` means the choice opens no submenu.

In [6]:
nav_menus = {
    "client_management": {
        "title": "Client Management Menu",
        "options": [
            {"menu_item": "Select Client", "next_menu": "client_workspace"},
            {"menu_item": "Create Client", "next_menu": None},
        ],
    },
    "client_workspace": {
        "title": "Client Workspace",
        "options": [
            {"menu_item": "Transactions", "next_menu": "transactions"},
            {"menu_item": "Reports", "next_menu": "reports"},
        ],
    },
    "transactions": {
        "title": "Transactions",
        "options": [
            {"menu_item": "Contribution", "next_menu": None},
            {"menu_item": "Withdrawal", "next_menu": None},
        ],
    },
    "reports": {
        "title": "Reports",
        "options": [
            {"menu_item": "Holdings", "next_menu": None},
            {"menu_item": "Performance", "next_menu": None},
        ],
    },
}

In [ ]:
# QUESTION 2
# A. What is the difference between the string "client_workspace"
#    and nav_menus["client_workspace"]?
# B. What does next_menu = None mean in this navigation-only system?

# YOUR ANSWERS:

---
## 3. Indirection: a name leads to a menu

An option doesn't *contain* the next menu. It contains the next menu's **name**, and we use that name to look the menu up. Using a name to find the real thing is **indirection**.

Follow the chain: **name → menu → option → next name → next menu**.

In [ ]:
from pprint import pprint as pp
menu_name = "client_management"                             # a string
menu = nav_menus[menu_name]                    # the menu dictionary
print("Menu name:      ", menu_name)
pp(menu, width=120, sort_dicts=False)


In [ ]:
for idx, option in enumerate(menu['options'], 1): # print menu items
    print(f"{idx}) {option['menu_item']}")


In [ ]:
print('\nUser enters 1\n')
choice = "1"                                   # what the user typed
option = menu["options"][int(choice) - 1]      # choices are 1-based; lists are 0-based
print("Selected:       ", option["menu_item"])

In [ ]:
next_menu_name = option["next_menu"]           # another string
print("Next menu name: ", next_menu_name)



In [ ]:
next_menu = nav_menus[next_menu_name]          # look it up the same way
print("Next menu title:", next_menu["title"])

In [ ]:
# QUESTION 3 — Predict, then verify
# What value and type does each expression produce?
#   nav_menus["client_workspace"]["options"][1]["next_menu"]
#   nav_menus["client_management"]["options"][1]["next_menu"]
# Write your predictions as comments, then check them with print() and type().

# YOUR PREDICTIONS AND CODE:

---
## 4. The stack remembers the route

A single variable can tell us the current menu, but not how we got there. The stack stores the **names** of the menus we've passed through. The top (`stack[-1]`) is the current menu.

In [ ]:
stack = ["client_management"]
print("Start:              ", stack)

# User chooses 1) Select Client
option = nav_menus[stack[-1]]["options"][0]
stack.append(option["next_menu"])
print("After Select Client:", stack)

# User chooses 2) Reports
option = nav_menus[stack[-1]]["options"][1]
stack.append(option["next_menu"])
print("After Reports:      ", stack)

# User presses Enter to go back
stack.pop()
print("After Enter:        ", stack)
print("Current menu:       ", nav_menus[stack[-1]]["title"])

In [ ]:
# QUESTION 4 — Trace by hand (predict; you'll check this in Section 5)
# Start: stack = ["client_management"]
# Choices: Select Client -> Transactions -> Contribution -> Enter
# A. Write the full stack after EACH choice.
# B. Why does choosing Contribution leave the stack unchanged?
# C. After Enter, which menu is current, and what in the stack tells the program that?

# YOUR ANSWERS:

---
## 5. One loop for every menu

Every pass through the loop does the same three things:

1. **Retrieve** the current menu from the name on top of the stack: `current_menu = menus[stack[-1]]`
2. **Select** an option from that menu: `option = current_menu["options"][int(choice) - 1]`
3. **Store** the destination's *name* if there is one: `stack.append(option["next_menu"])`

Step 3 stores a string; it does **not** retrieve the next menu. That lookup happens at step 1 on the **next** pass. Enter pops the stack; when the stack is empty, navigation ends.

Compare this to Section 1: there are no menu-specific branches at all.

In [ ]:
def display_nav_menu(current_menu):
    print("\n" + "=" * 36)
    print(current_menu["title"])
    print("=" * 36)
    for number, option in enumerate(current_menu["options"], 1):
        print(f"{number}) {option['menu_item']}")
    print("Press Enter to go back")


def run_navigation(menus):
    stack = ["client_management"]

    while stack:
        # 1. RETRIEVE: the name on top of the stack finds the menu (indirection).
        current_menu = menus[stack[-1]]
        display_nav_menu(current_menu)

        choice = input("Choice: ").strip()

        if choice == "":                  # go back
            stack.pop()
            continue

        if not choice.isdigit() or not 1 <= int(choice) <= len(current_menu["options"]):
            print("Invalid choice")
            continue

        # 2. SELECT: the numbered choice picks an option in this menu.
        option = current_menu["options"][int(choice) - 1]

        # 3. STORE: push the destination NAME; it is looked up next pass.
        if option["next_menu"] is not None:
            stack.append(option["next_menu"])
        else:
            print(f"\n{option['menu_item']}: no submenu — does nothing yet.")

    print("Navigation finished.")


# OPTIONAL: uncomment to navigate interactively.
# run_navigation(nav_menus)

### Check your Question 4 trace

This cell runs the same three steps with scripted choices and prints the stack each time: **Select Client → Transactions → Contribution → Enter → Enter → Enter**. Compare the first four iterations with your predictions.

In [ ]:
stack = ["client_management"]
test_choices = ["1", "1", "1", "", "", ""]

for step, choice in enumerate(test_choices, 1):
    current_menu = nav_menus[stack[-1]]                  # 1. RETRIEVE
    print(f"\nIteration {step} | menu: {current_menu['title']} | choice: {choice!r}")
    print("  Stack before:", stack)

    if choice == "":
        stack.pop()
    else:
        option = current_menu["options"][int(choice) - 1]   # 2. SELECT
        print("  Selected:", option["menu_item"], "-> next_menu:", option["next_menu"])
        if option["next_menu"] is not None:
            stack.append(option["next_menu"])               # 3. STORE

    print("  Stack after: ", stack)

print("\nStack empty — navigation ends.")

---
## 6. Extend PEMS by changing data only

Because menus are data, adding a menu means editing `nav_menus` — not the loop. A typo in menu data, though, won't show up until someone navigates to it, so we check the data first with a supplied validator. It confirms that every menu and option has the right keys and types, and that every `next_menu` names a menu that exists. You do not need to write it.

In [ ]:
# SUPPLIED VALIDATOR — read it, but you do not need to write or change it.
def validate_menus(menus):
    valid = True
    if not isinstance(menus, dict):
        print("Menu data must be a dictionary.")
        return False
    if "client_management" not in menus:
        print('Missing starting menu: "client_management"')
        valid = False
    for menu_name, menu in menus.items():
        if not isinstance(menu, dict):
            print(f"{menu_name}: menu definition must be a dictionary")
            valid = False
            continue
        if not isinstance(menu.get("title"), str) or not menu["title"].strip():
            print(f'{menu_name}: missing or invalid "title"')
            valid = False
        options = menu.get("options")
        if not isinstance(options, list) or not options:
            print(f'{menu_name}: "options" must be a non-empty list')
            valid = False
            continue
        for number, option in enumerate(options, 1):
            location = f"{menu_name}, option {number}"
            if not isinstance(option, dict):
                print(f"{location}: option must be a dictionary")
                valid = False
                continue
            label = option.get("menu_item")
            if not isinstance(label, str) or not label.strip():
                print(f'{location}: missing or invalid "menu_item"')
                valid = False
            if "next_menu" not in option:
                print(f'{location}: missing "next_menu"')
                valid = False
                continue
            next_menu = option["next_menu"]
            if next_menu is not None and next_menu not in menus:
                print(f"{location}: undefined next_menu: {next_menu}")
                valid = False
    if valid:
        print("Menu structures and submenu references are valid.")
    return valid


validate_menus(nav_menus)

### Guided exercise — Add Tax Reports

Add a menu with key `"tax_reports"`, title **Tax Reports**, and two choices: **Realized Gains** and **Dividend Income** (both `next_menu = None`). Then add a **Tax Reports** option to the existing Reports menu that links to it. Do **not** change `run_navigation()` or `display_nav_menu()`.

**Predict first:** Suppose you misspelled the link as `"tax_report"` and skipped validation. A user chooses **Select Client → Reports → Tax Reports**.
- Does the program crash when it *appends* the misspelled name, or later?
- Which line of `run_navigation()` raises the `KeyError`, and why there?

**Rerunning:** `append()` changes the existing list, so running your cell twice adds a duplicate option. To retry, use **Kernel → Restart & Run All**.

In [ ]:
# GUIDED EXERCISE — Add Tax Reports by changing menu data only.
#
# MY PREDICTION (where and why a misspelled link fails):
#
# MY CODE:

In [ ]:
# Check the guided exercise.
if validate_menus(nav_menus):
    display_nav_menu(nav_menus["reports"])

# Then test it: Select Client -> Reports -> Tax Reports -> Realized Gains,
# and use Enter to return.
# run_navigation(nav_menus)

---
## 7. Independent design — you decide what PEMS needs

Choose **one useful PEMS feature** that isn't in the menus yet. Decide where it belongs, its menu name, and at least two choices (terminal choices use `next_menu = None`). Sketch the menu tree first, then add your menu and link it from an existing menu. Do not change the navigation functions.

The validator checks structure and references, but it can't tell whether your menu is reachable where you intended. Only navigating it can, so the interactive test below is **required**.

In [ ]:
# INDEPENDENT DESIGN — plan first, implement second.
#
# MY MENU TREE SKETCH (indented comments):
#
# MY CODE:


validate_menus(nav_menus)

In [ ]:
# REQUIRED INTERACTIVE TEST
# Uncomment and run once: navigate from client_management to your new menu, choose one of
# its options, press Enter back to client_management, then Enter again to exit.
# Re-comment afterward so Restart & Run All stays noninteractive.
# run_navigation(nav_menus)

# MY TESTED ROUTE:
#
# DID IT WORK AS INTENDED?

---
## Takeaway

| Component | Responsibility |
|---|---|
| **Stack** | Remembers the route as menu names |
| **`nav_menus`** | Defines the menus, option labels, and destination names |
| **Lookup (indirection)** | Uses the name on top of the stack to retrieve the menu |
| **Navigation loop** | Retrieve, select, store — the same for every menu |

**Computational-thinking principle:** Identify what stays the same, represent what changes as data, and write one procedure that handles the differences.

**Looking ahead:** We reached Client Workspace without selecting a client, and Contribution without recording any money — menus hold destinations, not actions. Notebook 2 adds actions: functions that do real work and can sometimes keep the user on the current menu.